# logit-thermo 探索：チェスの記号を外した取り直し（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**。見込みは 20〜30 分。終わったら自動で切れます。
- 探索（予想なし）。前の下見の 250 局面（`A2_explore/chess_positions.json`）で、手の文字列から「+」と「#」を外した確率を 4 モデルで取り直し、メイトの手の確率が低かったのが記号のせいか手のせいかを比べて `A2_explore/chess_strip_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_explore_chess.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_chess_strip.py`（`7ea8d667cfa75060…`）は埋め込み。
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 チェスの記号を外した取り直し（メイトの # のせいか、4 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 チェスの記号を外した取り直し（メイトの # のせいか、4 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore chess strip', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import gc, math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCCH': '4d5c4b4a3cdba37ef27894bdc9281c2ffbd0a0679e27db5e721c53a48aac71f6'})
SRCCH = _s['SRCCH']
SRCST = '"""探索：メイトの手の確率がほぼ 0 だったのは「#」という記号のせいか、手そのもののせいか（2026-10-30 11:1x）。予想なし。GPU（L4）。\nscripts/a2_explore_chess.py（4d5c4b4a…）を実行したのと同じ名前空間で実行する（score_moves・load_any・CH_MODELS・CH_THEMES・spearman_np を使う）。\n\nオーナー（2026-10-30 11:09）：（オーナーの承認・発言。原文は非公開の記録にある）\nやり方：A2_explore/chess_positions.json の 250 局面で、各合法手の SAN から「+」と「#」を外した文字列の log 確率を取り直す。\n外した文字列は、そのあとに + や # が付く場合も含めた「その手の確率」になる（手の SAN は、+ と # を外しても、ほかの合法手の\nSAN の頭にはならない。ただし O-O と O-O-O は前と同じく前後の関係）。もとの（記号つきの）log 確率は chess_scores_*.json から読む。\n"""\nimport json\nimport numpy as np\n\n\ndef strip_san(s):\n    return s.rstrip(\'+#\')\n\n\ndef strip_summary(positions, orig, strip):\n    """orig・strip：{モデル: [局面ごとの log 確率の並び]} → テーマごとの比べ。"""\n    out = {}\n    for m in strip:\n        row = {}\n        for th in CH_THEMES + [\'all\']:\n            idx = [i for i, pz in enumerate(positions) if th == \'all\' or pz[\'theme\'] == th]\n            r = {}\n            for tag, S in ((\'orig\', orig[m]), (\'strip\', strip[m])):\n                pb, t1, rk, rho, mass = [], [], [], [], []\n                for i in idx:\n                    lp = np.asarray(S[i], float); E = np.asarray(positions[i][\'E\'], float)\n                    p = np.exp(lp - lp.max()); p /= p.sum()\n                    best = E <= 0.01\n                    pb.append(float(p[best].sum())); t1.append(float(best[int(np.argmax(p))]))\n                    order = np.argsort(-p)\n                    rk.append(int(np.flatnonzero(best[order])[0]) + 1)             # 最善手（いちばん良いもの）が何番目か\n                    rho.append(spearman_np(-lp, E)); mass.append(float(np.exp(lp).sum()))\n                r[tag] = dict(p_on_best=float(np.mean(pb)), top1_best=float(np.mean(t1)), best_rank_median=float(np.median(rk)),\n                              rho_median=float(np.nanmedian(rho)), legal_mass_mean=float(np.mean(mass)))\n            # 記号の付いた手（+ か #）にどれだけ確率が乗っているか（もと → 外した後）\n            chk_o, chk_s = [], []\n            for i in idx:\n                sans = positions[i][\'moves_san\']\n                mark = np.array([s.endswith(\'+\') or s.endswith(\'#\') for s in sans])\n                if mark.any() and (~mark).any():\n                    for S, acc in ((orig[m], chk_o), (strip[m], chk_s)):\n                        lp = np.asarray(S[i], float); p = np.exp(lp - lp.max()); p /= p.sum()\n                        acc.append(float(p[mark].sum() / mark.mean()))                 # 一様なら 1\n            r[\'check_ratio\'] = dict(orig=float(np.mean(chk_o)) if chk_o else None, strip=float(np.mean(chk_s)) if chk_s else None, n=len(chk_o))\n            row[th] = r\n        out[m] = row\n    return out\n\n\ndef self_test_strip():\n    assert strip_san(\'Qxf7#\') == \'Qxf7\' and strip_san(\'Nf3+\') == \'Nf3\' and strip_san(\'O-O\') == \'O-O\' and strip_san(\'e8=Q+\') == \'e8=Q\'\n    pos = [dict(theme=\'mateIn1\', E=[0.0, 0.5, 0.5], moves_san=[\'Qxf7#\', \'Nf3\', \'d4\'])]\n    orig = {\'m\': [[-20.0, -1.0, -1.0]]}; strip = {\'m\': [[-0.5, -1.0, -1.0]]}\n    s = strip_summary(pos, orig, strip)[\'m\'][\'mateIn1\']\n    assert s[\'orig\'][\'top1_best\'] == 0 and s[\'strip\'][\'top1_best\'] == 1 and s[\'strip\'][\'best_rank_median\'] == 1, s\n    assert s[\'check_ratio\'][\'n\'] == 1 and s[\'check_ratio\'][\'orig\'] < 0.01 and s[\'check_ratio\'][\'strip\'] > 1, s\n    return \'OK\'\n'
assert hashlib.sha256(SRCST.encode()).hexdigest() == '7ea8d667cfa750604a2f103d1807176691fed3a82feae7f7db1d9c8b25e26227'
exec(SRCCH)
exec(SRCST)
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('自己テスト：', self_test_strip())


In [ ]:
# セル3：取り直し → まとめ（結果は表示しない）
try:
    OUTD = '/content/drive/MyDrive/logit-thermo' + '/A2_explore'
    P = json.load(open(OUTD + '/chess_positions.json'))['positions']
    assert len(P) == 250, len(P)
    orig, strip = {}, {}
    for name, hf_id, dt in CH_MODELS:
        orig[name] = json.load(open(OUTD + f'/chess_scores_{name}.json'))['scores']
        sp = OUTD + f'/chess_scores_strip_{name}.json'
        if not os.path.exists(sp):
            t0 = time.time()
            tok, model = load_any(hf_id, dt, 'cuda')
            S = [score_moves(tok, model, pz['prompt'], [strip_san(s) for s in pz['moves_san']], 'cuda').tolist() for pz in P]
            json.dump(dict(model=hf_id, dtype=dt, scores=S), open(sp + '.part', 'w')); os.replace(sp + '.part', sp)
            del model; gc.collect(); torch.cuda.empty_cache()
            notify(f'✅ {name} おわり（{time.time() - t0:.0f} 秒）', tags='white_check_mark')
        strip[name] = json.load(open(sp))['scores']
    summ = strip_summary(P, orig, strip)
    summ = dict(models=summ, a2_explore_chess_sha256='4d5c4b4a3cdba37ef27894bdc9281c2ffbd0a0679e27db5e721c53a48aac71f6', a2_explore_chess_strip_sha256='7ea8d667cfa750604a2f103d1807176691fed3a82feae7f7db1d9c8b25e26227')
    json.dump(_clean(summ), open(OUTD + '/chess_strip_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 チェスの記号を外した取り直しで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：記号を外した取り直しを A2_explore/chess_strip_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
